# Minesweeping — Jane Street, October 2014

https://www.janestreet.com/puzzles/minesweeping-index/

## Best Solution: 26/27

```
- - - -
- - 6 -
- 6 - -
- - - -
```




In [16]:
from fractions import Fraction
from itertools import product


def board_size(rows):
    """Return (height, width) of a board written as a list of row strings."""
    return len(rows), len(rows[0])


def parse_board(rows):
    """Return a dict mapping (row, column) to the number showing in that revealed square."""
    height, width = board_size(rows)
    for row_text in rows:
        if len(row_text) != width:
            raise ValueError("every row must be the same length")

    clues = {}
    for row_index, row_text in enumerate(rows):
        for column_index, character in enumerate(row_text):
            if character == "-":
                continue
            if not character.isdigit():
                raise ValueError(
                    "a square must be '-' or a single digit, got " + repr(character)
                )
            clues[(row_index, column_index)] = int(character)
    return clues

In [17]:
def neighbours(cell, height, width):
    """Return the list of in-bounds squares touching `cell`, diagonals included."""
    row, column = cell
    found = []
    for row_offset in (-1, 0, 1):
        for column_offset in (-1, 0, 1):
            is_the_cell_itself = row_offset == 0 and column_offset == 0
            if is_the_cell_itself:
                continue

            neighbour_row = row + row_offset
            neighbour_column = column + column_offset
            inside_rows = 0 <= neighbour_row < height
            inside_columns = 0 <= neighbour_column < width
            if inside_rows and inside_columns:
                found.append((neighbour_row, neighbour_column))
    return found


def unrevealed_cells(rows):
    """Return the squares showing no number — the ones free to hold a mine."""
    height, width = board_size(rows)
    clues = parse_board(rows)

    hidden = []
    for row_index in range(height):
        for column_index in range(width):
            if (row_index, column_index) not in clues:
                hidden.append((row_index, column_index))
    return hidden

In [18]:
def clues_are_satisfied(clues, mines, height, width):
    """Return True if every revealed number equals the count of mines in the squares around it."""
    for clue_cell, clue_number in clues.items():
        adjacent_mines = 0
        for neighbour in neighbours(clue_cell, height, width):
            if neighbour in mines:
                adjacent_mines += 1
        if adjacent_mines != clue_number:
            return False
    return True


def solve(rows):
    """Return every mine placement consistent with the board, as a list of frozensets of cells.

    A revealed square is never a mine, so only the unrevealed squares are free. On a 4x4 board
    that is at most 2**16 placements — small enough to test every one, so there is nothing to
    gain from propagating constraints instead.
    """
    height, width = board_size(rows)
    clues = parse_board(rows)
    hidden = unrevealed_cells(rows)

    solutions = []
    for assignment in product([False, True], repeat=len(hidden)):
        mines = set()
        for cell, holds_a_mine in zip(hidden, assignment):
            if holds_a_mine:
                mines.add(cell)

        if clues_are_satisfied(clues, mines, height, width):
            solutions.append(frozenset(mines))
    return solutions

In [19]:
def render(rows, mines):
    """Return the filled-in board as text lines: the clue digit, '*' for a mine, '.' for a safe square."""
    height, width = board_size(rows)
    clues = parse_board(rows)

    lines = []
    for row_index in range(height):
        symbols = []
        for column_index in range(width):
            cell = (row_index, column_index)
            if cell in clues:
                symbols.append(str(clues[cell]))
            elif cell in mines:
                symbols.append("*")
            else:
                symbols.append(".")
        lines.append(" ".join(symbols))
    return lines


def print_board(rows):
    """Print the board as given, before any mines are placed."""
    print("board:")
    for row_text in rows:
        print("   " + " ".join(row_text))


def print_solutions(rows, limit=None):
    """Print every consistent mine placement for the board, and return the list of them."""
    solutions = solve(rows)

    print_board(rows)
    print()
    print(str(len(solutions)) + " solution(s)")
    print()

    shown = solutions
    if limit is not None:
        shown = solutions[:limit]

    for number, mines in enumerate(shown, start=1):
        print("--- solution " + str(number) + "  (" + str(len(mines)) + " mines) ---")
        for line in render(rows, mines):
            print("   " + line)
        print()

    if len(shown) < len(solutions):
        print("... " + str(len(solutions) - len(shown)) + " more not shown")

    return solutions

In [20]:
def mine_probabilities(rows):
    """Return {(row, column): P(S)} as exact fractions, or an empty dict if the board has no solutions."""
    solutions = solve(rows)
    if len(solutions) == 0:
        return {}

    probabilities = {}
    for cell in unrevealed_cells(rows):
        appearances = 0
        for mines in solutions:
            if cell in mines:
                appearances += 1
        probabilities[cell] = Fraction(appearances, len(solutions))
    return probabilities


def report(rows):
    """Print the board, its solution count, and every unrevealed square's P(S), highest first."""
    print_board(rows)
    print()

    probabilities = mine_probabilities(rows)
    if len(probabilities) == 0:
        print("no solutions — this board is impossible, so P(S) is undefined")
        return

    solution_count = len(solve(rows))
    print(str(solution_count) + " solution(s)")
    print()

    # sort by probability, highest first
    ranked = sorted(probabilities.items(), key=lambda entry: entry[1], reverse=True)

    print("P(S) per unrevealed square:")
    for cell, probability in ranked:
        note = ""
        if probability == 1:
            note = "   (certain — not eligible)"
        print(
            "   "
            + str(cell)
            + "   "
            + str(probability)
            + "   "
            + format(float(probability), ".4f")
            + note
        )

    best_cell = None
    best_probability = None
    for cell, probability in ranked:
        if probability < 1:
            best_cell = cell
            best_probability = probability
            break

    print()
    if best_cell is None:
        print("no square has P(S) < 1")
    else:
        print(
            "best eligible square: "
            + str(best_cell)
            + "   P(S) = "
            + str(best_probability)
        )

## First attempt 7 and two 4s

A 7 will have 7/8th naturally, and by putting 4 nears the cell of interest (1,2), we can further enumerate solutions where (1,2) is a mine while keeping only 1 solution where it isn't.
```
- - 4 -
- - - -
- 7 - 4
- - - -
```


In [24]:
## First attempt 7 and two 4s
SEVEN_FOUR_FOUR = [
    "--4-",
    "----",
    "-7-4",
    "----",
]

report(SEVEN_FOUR_FOUR)

board:
   - - 4 -
   - - - -
   - 7 - 4
   - - - -

36 solution(s)

P(S) per unrevealed square:
   (1, 1)   17/18   0.9444
   (1, 2)   17/18   0.9444
   (2, 2)   8/9   0.8889
   (3, 2)   8/9   0.8889
   (1, 0)   5/6   0.8333
   (2, 0)   5/6   0.8333
   (3, 0)   5/6   0.8333
   (3, 1)   5/6   0.8333
   (1, 3)   7/9   0.7778
   (0, 1)   2/3   0.6667
   (0, 3)   2/3   0.6667
   (0, 0)   1/2   0.5000
   (3, 3)   1/2   0.5000

best eligible square: (1, 1)   P(S) = 17/18


## Second attempt 7 and one 4
Weirdly I think the second 4 hurts us.  It actually restricted the solution set without giving us much additional -- once (1,2) is empty it doesn't do anything the other 4 doesn't already do, and it prunes some valid solutions from our set when (1,2) is a mine.

```
- - 4 -
- - - -
- 7 - -
- - - -
```

In [ ]:
SEVEN_FOUR = [
    "--4-",
    "----",
    "-7--",
    "----",
]
report(SEVEN_FOUR)

board:
   - - 4 -
   - - - -
   - 7 - -
   - - - -

160 solution(s)

P(S) per unrevealed square:
   (1, 1)   19/20   0.9500
   (1, 2)   19/20   0.9500
   (1, 0)   17/20   0.8500
   (2, 0)   17/20   0.8500
   (2, 2)   17/20   0.8500
   (3, 0)   17/20   0.8500
   (3, 1)   17/20   0.8500
   (3, 2)   17/20   0.8500
   (0, 1)   7/10   0.7000
   (0, 3)   7/10   0.7000
   (1, 3)   7/10   0.7000
   (0, 0)   1/2   0.5000
   (2, 3)   1/2   0.5000
   (3, 3)   1/2   0.5000

best eligible square: (1, 1)   P(S) = 19/20
board:
   - - - -
   - - 6 -
   - 6 - -
   - - - -

108 solution(s)

P(S) per unrevealed square:
   (1, 1)   26/27   0.9630
   (2, 2)   26/27   0.9630
   (0, 1)   22/27   0.8148
   (0, 2)   22/27   0.8148
   (0, 3)   22/27   0.8148
   (1, 0)   22/27   0.8148
   (1, 3)   22/27   0.8148
   (2, 0)   22/27   0.8148
   (2, 3)   22/27   0.8148
   (3, 0)   22/27   0.8148
   (3, 1)   22/27   0.8148
   (3, 2)   22/27   0.8148
   (0, 0)   1/2   0.5000
   (3, 3)   1/2   0.5000

best eligible squ

## Best attempt: 26/27

The 7 and 4 solution got us to 19/20.  My best solution from playing around came from just doing 6-6.  

```
- - - -
- - 6 -
- 6 - -
- - - -
```

This gives us symettry on the 2 shared squares, and if one of them is free it sets the rest to be mines. If (1,1) is NOT a mine, the solution is set.  However, if it IS a mine, then if (2,2) is not a mine, then our solution is set.  Howver, if (2,2) is a mine, we have 5 options for not a mine for hte first six, and 5 options for the second, a total of 1 + 1 + 5*5 = 27 boards.  Thus the chance that (1,1) or (2,2) is a mine is 26/27.  

In [25]:
SIX_SIX = [
    "----",
    "--6-",
    "-6--",
    "----",
]

report(SIX_SIX)

board:
   - - - -
   - - 6 -
   - 6 - -
   - - - -

108 solution(s)

P(S) per unrevealed square:
   (1, 1)   26/27   0.9630
   (2, 2)   26/27   0.9630
   (0, 1)   22/27   0.8148
   (0, 2)   22/27   0.8148
   (0, 3)   22/27   0.8148
   (1, 0)   22/27   0.8148
   (1, 3)   22/27   0.8148
   (2, 0)   22/27   0.8148
   (2, 3)   22/27   0.8148
   (3, 0)   22/27   0.8148
   (3, 1)   22/27   0.8148
   (3, 2)   22/27   0.8148
   (0, 0)   1/2   0.5000
   (3, 3)   1/2   0.5000

best eligible square: (1, 1)   P(S) = 26/27
